# LightGBM 多因子选股策略

## 论文参考

- **标题**: Research on Machine Learning Driven Stock Selection Strategy
- **作者**: Kerang Wang
- **年份**: 2023
- **DOI**: 10.54254/2754-1169/49/20230529

### 摘要

本文研究了基于机器学习的量化选股策略，采用 LightGBM 梯度提升模型对 A 股市场进行截面收益预测。
通过构建包含动量、波动率、技术指标等多维因子体系，使用滚动窗口训练模型并月度调仓，
策略在回测期内取得了 **年化 69.33%** 的收益率，显著超越基准指数。

> **⚠️ 教学演示声明**
> 
> 本 Notebook 为量化策略算法教学样例，回测结果包含**前视偏差 (Look-ahead Bias)**：
> - 训练标签使用了未来收益率（`shift(-N)`）
> - StandardScaler 等预处理可能在全量数据上拟合
> - 模型参数选择可能基于完整样本期
> 
> **所有回测收益率仅供学习参考，不代表策略的实际可期望表现，不可直接用于实盘交易。**

## 算法原理

### LightGBM 梯度提升框架

LightGBM 是一种高效的梯度提升决策树 (GBDT) 实现，通过以下步骤逐步构建集成模型：

**1. 目标函数**

$$\mathcal{L}(\phi) = \sum_{i=1}^{n} l(y_i, \hat{y}_i) + \sum_{k=1}^{K} \Omega(f_k)$$

其中 $l$ 是损失函数，$\Omega(f_k) = \gamma T + \frac{1}{2}\lambda \|w\|^2$ 是正则化项。

**2. 逐轮迭代拟合残差**

第 $t$ 轮的预测为：
$$\hat{y}_i^{(t)} = \hat{y}_i^{(t-1)} + \eta \cdot f_t(x_i)$$

其中 $\eta$ 是学习率，$f_t$ 是第 $t$ 棵树拟合的负梯度方向：
$$f_t(x_i) \approx -\frac{\partial l(y_i, \hat{y}_i^{(t-1)})}{\partial \hat{y}_i^{(t-1)}}$$

**3. LightGBM 的核心优化**

- **Histogram-based splitting**: 将连续特征离散化为直方图，加速分裂点搜索
- **GOSS (Gradient-based One-Side Sampling)**: 保留大梯度样本，随机采样小梯度样本
- **EFB (Exclusive Feature Bundling)**: 将互斥特征捆绑，减少特征维度

**4. 选股逻辑**

- 每月末用过去 12 个月数据训练模型
- 预测下月各股票截面收益率
- 选择预测收益最高的 Top-10 股票等权持有

In [ ]:
# ============================================================
# Cell 3: 动画展示 - 梯度提升逐轮拟合过程
# ============================================================
import numpy as np
import plotly.graph_objects as go
from sklearn.tree import DecisionTreeRegressor

np.random.seed(42)
X_demo = np.sort(np.random.uniform(0, 10, 200))
y_demo = np.sin(X_demo) + 0.3 * np.cos(3 * X_demo) + np.random.normal(0, 0.3, 200)

n_rounds = 12
learning_rate = 0.15
residuals = y_demo.copy()
prediction = np.zeros_like(y_demo)
frames = []

# Round 0
frames.append(go.Frame(
    data=[
        go.Scatter(x=X_demo, y=y_demo, mode='markers', name='真实值',
                   marker=dict(color='#2196F3', size=4, opacity=0.5)),
        go.Scatter(x=X_demo, y=prediction, mode='lines', name='集成预测',
                   line=dict(color='#F44336', width=3)),
        go.Scatter(x=X_demo, y=residuals, mode='markers', name='残差',
                   marker=dict(color='#FF9800', size=3, opacity=0.4)),
    ],
    name='Round 0'
))

for r in range(1, n_rounds + 1):
    tree = DecisionTreeRegressor(max_depth=3, random_state=r)
    tree.fit(X_demo.reshape(-1, 1), residuals)
    pred_r = tree.predict(X_demo.reshape(-1, 1))
    prediction += learning_rate * pred_r
    residuals = y_demo - prediction
    mse = np.mean(residuals ** 2)

    frames.append(go.Frame(
        data=[
            go.Scatter(x=X_demo, y=y_demo, mode='markers', name='真实值',
                       marker=dict(color='#2196F3', size=4, opacity=0.5)),
            go.Scatter(x=X_demo, y=prediction, mode='lines', name='集成预测',
                       line=dict(color='#F44336', width=3)),
            go.Scatter(x=X_demo, y=residuals, mode='markers', name='残差',
                       marker=dict(color='#FF9800', size=3, opacity=0.4)),
        ],
        name=f'Round {r} (MSE={mse:.4f})'
    ))

fig = go.Figure(
    data=frames[0].data,
    frames=frames,
    layout=go.Layout(
        title=dict(text='LightGBM: 梯度提升逐轮拟合残差'),
        xaxis_title='特征 X', yaxis_title='目标 Y',
        height=500, width=900,
        updatemenus=[dict(type='buttons', buttons=[
            dict(label='▶ 播放', method='animate',
                 args=[None, {'frame': {'duration': 600}, 'transition': {'duration': 300}}]),
            dict(label='⏸ 暂停', method='animate',
                 args=[[None], {'frame': {'duration': 0}, 'mode': 'immediate'}]),
        ])],
        sliders=[dict(
            steps=[dict(args=[[f.name], {'frame': {'duration': 0}, 'mode': 'immediate'}],
                        label=f.name, method='animate') for f in frames],
            active=0,
        )],
    )
)
fig.show()

In [ ]:
# ============================================================
# Cell 4: 环境设置与导入
# ============================================================
import sys
import os
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
from datetime import datetime

warnings.filterwarnings('ignore')
np.random.seed(42)

# 导入共享模块
sys.path.insert(0, '..')
from shared.data_fetcher import get_stock_daily, get_index_daily, get_multiple_stocks_daily
from shared.backtest_engine import MultiStockBacktester
from shared.factors import (
    momentum, volatility, rsi, macd, bollinger_bands, atr, adx, cci,
    turnover_factor, volume_price_corr, high_low_range, price_to_ma,
    winsorize, standardize
)
from shared.visualization import (
    plot_equity_curve, plot_drawdown, plot_metrics_table,
    plot_factor_importance, plot_returns_distribution
)
from shared.constants import DEFAULT_START, DEFAULT_END, CSI300_CODE

print('所有模块导入成功')
print(f'回测区间: {DEFAULT_START} - {DEFAULT_END}')

In [ ]:
# ============================================================
# Cell 5: 数据获取
# ============================================================

# 20只代表性CSI300成分股 (硬编码确保可靠性)
STOCK_POOL = [
    '600519',  # 贵州茅台
    '601318',  # 中国平安
    '600036',  # 招商银行
    '000858',  # 五粮液
    '601166',  # 兴业银行
    '600276',  # 恒瑞医药
    '601398',  # 工商银行
    '600030',  # 中信证券
    '000333',  # 美的集团
    '002415',  # 海康威视
    '600900',  # 长江电力
    '601888',  # 中国中免
    '000568',  # 泸州老窖
    '002304',  # 洋河股份
    '601012',  # 隆基绿能
    '600031',  # 三一重工
    '603259',  # 药明康德
    '600585',  # 海螺水泥
    '000661',  # 长春高新
    '601668',  # 中国建筑
]

try:
    stock_data = get_multiple_stocks_daily(STOCK_POOL, DEFAULT_START, DEFAULT_END)
    print(f'成功获取 {len(stock_data)} 只股票数据')
except Exception as e:
    print(f'数据获取异常: {e}')
    print('使用模拟数据...')
    dates = pd.bdate_range(DEFAULT_START, DEFAULT_END)
    stock_data = {}
    for sym in STOCK_POOL:
        np.random.seed(hash(sym) % 2**31)
        price = 50 + np.cumsum(np.random.randn(len(dates)) * 0.5)
        price = np.maximum(price, 5)
        stock_data[sym] = pd.DataFrame({
            'open': price * (1 + np.random.randn(len(dates)) * 0.005),
            'close': price,
            'high': price * (1 + np.abs(np.random.randn(len(dates)) * 0.01)),
            'low': price * (1 - np.abs(np.random.randn(len(dates)) * 0.01)),
            'volume': np.random.randint(100000, 5000000, len(dates)).astype(float),
            'turnover': np.random.uniform(0.3, 5.0, len(dates)),
        }, index=dates)

# 获取基准指数
try:
    benchmark = get_index_daily(CSI300_CODE, DEFAULT_START, DEFAULT_END)
    benchmark_prices = benchmark['close']
    print(f'基准数据: {len(benchmark_prices)} 条')
except Exception as e:
    print(f'基准获取异常: {e}, 使用模拟基准')
    dates = pd.bdate_range(DEFAULT_START, DEFAULT_END)
    benchmark_prices = pd.Series(
        5000 + np.cumsum(np.random.randn(len(dates)) * 10), index=dates
    )

In [ ]:
# ============================================================
# Cell 6: 因子工程
# ============================================================

def build_features_for_stock(df):
    """为单只股票构建全部因子"""
    features = pd.DataFrame(index=df.index)

    # 动量因子 (5/10/20日)
    mom = momentum(df['close'], periods=[5, 10, 20])
    features = pd.concat([features, mom], axis=1)

    # 波动率因子 (10/20日)
    vol = volatility(df['close'], windows=[10, 20])
    features = pd.concat([features, vol], axis=1)

    # RSI
    features['rsi_14'] = rsi(df['close'], 14)

    # MACD 直方图
    macd_df = macd(df['close'])
    features['macd_hist'] = macd_df['histogram']

    # 布林带 %B
    bb = bollinger_bands(df['close'], 20)
    features['bb_pctb'] = bb['bb_pctb']
    features['bb_width'] = bb['bb_width']

    # 换手率
    if 'turnover' in df.columns:
        features['turnover'] = df['turnover']
        features['turnover_ma5'] = df['turnover'].rolling(5).mean()
        features['turnover_ma10'] = df['turnover'].rolling(10).mean()

    # 量价相关性
    if 'volume' in df.columns:
        features['vp_corr_20'] = volume_price_corr(df['close'], df['volume'], 20)

    # 价格偏离均线
    ptm = price_to_ma(df['close'], windows=[5, 10, 20])
    features = pd.concat([features, ptm], axis=1)

    # ATR
    if all(c in df.columns for c in ['high', 'low']):
        features['atr_14'] = atr(df['high'], df['low'], df['close'], 14)
        features['hl_range'] = high_low_range(df['high'], df['low'], df['close'])

    return features


# 构建全部股票的因子面板
all_features = []
for sym, df in stock_data.items():
    if len(df) < 60:
        continue
    feats = build_features_for_stock(df)
    # 未来20日收益率作为标签
    feats['fwd_return_20d'] = df['close'].pct_change(20).shift(-20)
    feats['symbol'] = sym
    all_features.append(feats)

panel = pd.concat(all_features)
panel = panel.reset_index()
if 'index' in panel.columns:
    panel.rename(columns={'index': 'date'}, inplace=True)

# 定义特征列
FEATURE_COLS = [c for c in panel.columns if c not in ['date', 'symbol', 'fwd_return_20d']]
print(f'因子面板: {panel.shape[0]} 行 x {len(FEATURE_COLS)} 个因子')
print(f'因子列表: {FEATURE_COLS}')

In [ ]:
# ============================================================
# Cell 7: LightGBM 模型训练 (滚动窗口)
# ============================================================

# 滚动窗口参数
TRAIN_MONTHS = 12  # 训练窗口
TOP_K = 10         # 每月选股数量

# 获取所有月末日期
panel['date'] = pd.to_datetime(panel['date'])
panel['year_month'] = panel['date'].dt.to_period('M')
months = sorted(panel['year_month'].unique())

print(f'数据覆盖 {len(months)} 个月: {months[0]} ~ {months[-1]}')

# LightGBM 参数
lgb_params = {
    'objective': 'regression',
    'metric': 'mse',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'lambda_l1': 0.1,
    'lambda_l2': 0.1,
    'verbose': -1,
    'seed': 42,
}

selections = {}  # {date: [symbols]}
all_importances = []

for i in range(TRAIN_MONTHS, len(months) - 1):
    # 训练集: 过去12个月
    train_months_range = months[i - TRAIN_MONTHS: i]
    test_month = months[i]

    train_data = panel[panel['year_month'].isin(train_months_range)].copy()
    test_data = panel[panel['year_month'] == test_month].copy()

    # 清洗数据
    train_data = train_data.dropna(subset=FEATURE_COLS + ['fwd_return_20d'])
    test_data_clean = test_data.dropna(subset=FEATURE_COLS)

    if len(train_data) < 50 or len(test_data_clean) < 5:
        continue

    X_train = train_data[FEATURE_COLS].values
    y_train = train_data['fwd_return_20d'].values
    X_test = test_data_clean[FEATURE_COLS].values

    # 截面标准化
    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # 训练 LightGBM
    dtrain = lgb.Dataset(X_train, label=y_train)
    model = lgb.train(
        lgb_params, dtrain,
        num_boost_round=200,
    )

    # 预测并选股
    pred = model.predict(X_test)
    test_data_clean = test_data_clean.copy()
    test_data_clean['pred_return'] = pred

    # 取最后一天的截面作为选股依据
    last_day = test_data_clean.groupby('symbol')['pred_return'].last()
    top_stocks = last_day.nlargest(TOP_K).index.tolist()

    # 记录调仓日
    rebal_date = test_data_clean['date'].max()
    selections[rebal_date] = top_stocks

    # 记录特征重要性
    imp = dict(zip(FEATURE_COLS, model.feature_importance(importance_type='gain')))
    all_importances.append(imp)

    if (i - TRAIN_MONTHS) % 6 == 0:
        print(f'  {test_month}: 选股 {top_stocks[:5]}...')

print(f'\n共生成 {len(selections)} 期选股结果')

# 平均特征重要性
avg_importance = {}
for col in FEATURE_COLS:
    vals = [imp.get(col, 0) for imp in all_importances]
    avg_importance[col] = np.mean(vals)

print('\nTop-10 重要因子:')
for k, v in sorted(avg_importance.items(), key=lambda x: -x[1])[:10]:
    print(f'  {k}: {v:.2f}')

In [ ]:
# ============================================================
# Cell 8: 信号生成与回测
# ============================================================

# 构建价格字典 {symbol: close_series}
all_prices = {}
for sym, df in stock_data.items():
    all_prices[sym] = df['close']

# 多股票组合回测
backtester = MultiStockBacktester(
    initial_capital=1_000_000,
    rebalance_freq='M'
)

result = backtester.run(
    all_prices=all_prices,
    selections=selections,
    benchmark_prices=benchmark_prices
)

print('=== LightGBM 选股策略回测结果 ===')
for k, v in result['metrics'].items():
    print(f'  {k}: {v}')

In [ ]:
# ============================================================
# Cell 9: 绩效可视化
# ============================================================

# 1. 收益曲线
benchmark_equity = None
if result.get('benchmark_returns') is not None:
    benchmark_equity = (1 + result['benchmark_returns']).cumprod() * result['equity_curve'].iloc[0]
plot_equity_curve(result['equity_curve'], benchmark_equity,
                  title='LightGBM 多因子选股 - 累计收益')

# 2. 回撤图
plot_drawdown(result['equity_curve'],
              title='LightGBM 多因子选股 - 回撤')

# 3. 因子重要性
plot_factor_importance(avg_importance,
                       title='LightGBM 因子重要性 (平均 Gain)',
                       top_n=15)

# 4. 收益率分布
plot_returns_distribution(result['returns'],
                          title='LightGBM 策略日收益率分布')

# 5. 绩效表
plot_metrics_table(result['metrics'],
                   title='LightGBM 选股策略绩效指标')

## 结果讨论

### 策略表现

1. **LightGBM 的优势**: 在高维因子空间中能高效捕捉非线性关系，训练速度远优于传统 GBDT
2. **因子重要性**: 动量因子 (mom_5, mom_10) 和波动率因子 (vol_10, vol_20) 通常排名靠前，
   说明短期动量和波动率对未来收益有较强预测力
3. **滚动训练**: 12 个月滚动窗口在捕捉市场变化和保证样本量之间取得平衡

### 局限性

- 使用 20 只代表性股票简化了实际 CSI300 全样本选股场景
- 未考虑停牌、涨跌停等交易限制
- 等权配置未考虑风险预算或组合优化
- 回测区间较短，策略稳健性需更长周期验证

### 改进方向

- 加入基本面因子 (PB/PE/ROE)
- 使用 LightGBM 的 `lambdarank` 目标函数替代回归
- 结合行业中性化和市值中性化
- 集成多个时间尺度的模型预测